# 05 — MCMC Ensemble (Post-Rucho State-Court Strategy)

**Persona:** plaintiff's expert in a state-court partisan gerrymandering case.
**Time:** ≤ 1800 seconds (nightly only — too slow for PR CI).
**Outcome:** generate an N≥1000 ensemble of partisan-blind plans, validate convergence (R-hat / ESS / Hamming), report the enacted plan's percentile rank for partisan metrics.

This is the load-bearing notebook for the post-Rucho strategy documented in `docs/legal/FAIRNESS_DOCTRINE.md` §1.5 + §3 + §5. The ensemble + diagnostics + percentile-rank output is the evidence package state courts (PA, NC, NY, NM) have accepted.

**Note:** the ensemble GENERATION step requires the GerryChain wrapper at `scripts/research/mcmc_ensemble.py` (deferred). The diagnostics math the notebook consumes (`redist-analysis::ensemble_diagnostics`) IS shipped today.

**Cross-references:**
- `docs/legal/FAIRNESS_DOCTRINE.md` §5 — Rucho "judicially manageable standard" addressed
- `redist-analysis::ensemble_diagnostics` — R-hat, ESS, Hamming autocorrelation (shipped)
- Plan Task 6 — MCMC wrapper around GerryChain (deferred)
- Plan Task 5 — `redist research validate-ensemble` percentile-rank CLI (deferred)


In [ ]:
from packaging.specifiers import SpecifierSet
REDIST_PY_RANGE = '>=0.4,<0.5'
GERRYCHAIN_RANGE = '>=0.3.2,<0.4'

import redist_py
assert redist_py.__version__ in SpecifierSet(REDIST_PY_RANGE)

try:
    import gerrychain
    assert gerrychain.__version__ in SpecifierSet(GERRYCHAIN_RANGE)
    print(f'redist_py {redist_py.__version__}, gerrychain {gerrychain.__version__} OK')
except ImportError:
    print('SKIP: gerrychain not installed; this notebook requires it for ensemble generation')
    raise SystemExit(0)


## TODO — notebook body

Sketch:

```python
# 1. Generate ensemble (Task 6: scripts/research/mcmc_ensemble.py — deferred).
from scripts.research.mcmc_ensemble import generate_ensemble  # noqa
ensemble = generate_ensemble(
    state='NC', year=2020,
    n_steps=10000, n_chains=4, seed=42,
    output_dir='outputs/v1/ensembles/nc_2020_neutral',
)

# 2. Validate convergence (math is shipped; CLI wrapper deferred).
from scripts.research.diagnostics import compute_diagnostics  # noqa
diag = compute_diagnostics('outputs/v1/ensembles/nc_2020_neutral')
for metric, rec in diag['rhat'].items():
    assert rec['r_hat'] < 1.05, f'{metric} R-hat {rec["r_hat"]} not converged'
for metric, rec in diag['ess'].items():
    assert rec['ess'] > 100, f'{metric} ESS {rec["ess"]} too low'

# 3. Compare enacted plan to ensemble (Task 5: validate-ensemble CLI — deferred).
import subprocess, json
subprocess.check_call([
    'redist', 'research', 'validate-ensemble',
    '--plan-label', 'nc_enacted_2022',
    '--ensemble-label', 'nc_2020_neutral',
])
with open('outputs/v1/ensembles/nc_2020_neutral/target_plan_percentiles.json') as f:
    pcts = json.load(f)
print(f"Enacted plan partisan-bias percentile: {pcts['partisan_bias']['percentile']}")
# A 99th+ percentile is the smoking gun for state-court intentional-gerrymandering claims.
```

The diagnostics module (`redist-analysis::ensemble_diagnostics`) implements R-hat, ESS, and Hamming autocorrelation TODAY — the three Python wrappers above (`generate_ensemble`, `compute_diagnostics`, `validate-ensemble`) are the deferred connectors. Once they ship, this notebook is the publication-quality evidence-package generator for state-court litigation.


In [ ]:
print('notebook completed within budget')
